In [0]:
from cleaning_functions import rename_columns, gender_fix, marital_status_fix, location_fix, trim_text
from pyspark.sql.functions import col, try_to_date, regexp_replace, substring
spark.sql("USE CATALOG biking_product_sales_lakehouse")
spark.sql("USE SCHEMA bronze")

In [0]:
# Reading all tables to a dataframe
df_customer_crm = spark.read.table("customer_crm")
df_customer_erp = spark.read.table("customer_erp")
df_location = spark.read.table("location_erp")
df_product_category = spark.read.table("product_category_subcat_erp")
df_product_crm = spark.read.table("product_crm")
df_sales_crm = spark.read.table("sales_crm")

dfs = {"customer_crm": df_customer_crm,
       "customer_erp": df_customer_erp,
       "customer_location": df_location,
       "product_cat_subcat": df_product_category,
       "product_crm": df_product_crm,
       "sales_crm": df_sales_crm
    }

In [0]:
# Renaming columns
column_names = {
    "customer_crm": {
        "cst_id": "customer_id",
        "cst_key": "customer_key",
        "cst_firstname": "firstname",
        "cst_lastname": "lastname",
        "cst_marital_status": "marital_status",
        "cst_gndr": "gender",
        "cst_create_date": "creation_date"
    },

    "customer_erp": {
        "CID": "customer_key",
        "BDATE": "birth_date",
        "GEN": "gender"
    },

    "customer_location": {
        "CID": "customer_key",
        "CNTRY": "country"
    },

    #prd_start_dt is wrongly named: It contains the values of end_date because the date values comes after the values of start_dt
    "product_crm": {
        "prd_id": "product_id",
        "prd_key": "product_key",
        "prd_nm": "product_name",
        "prd_cost": "product_cost",
        "prd_line": "product_line",
        "prd_start_dt": "product_end_date",
        "prd_end_dt": "product_start_date"
    },

    "product_cat_subcat": {
        "ID": "category_id",
        "CAT": "category",
        "SUBCAT": "sub_category",
        "MAINTENANCE": "maintenance"
    },

    "sales_crm": {
        "sls_ord_num": "order_id",
        "sls_prd_key": "product_key",
        "sls_cust_id": "customer_id",
        "sls_order_dt": "order_date",
        "sls_ship_dt": "ship_date",
        "sls_due_dt": "due_date",
        "sls_sales": "sales",
        "sls_quantity": "quantity",
        "sls_price": "price"
    }
}

dfs = {key: rename_columns(df, column_names[key]) for key, df in dfs.items()}


In [0]:
# Triming text columns

dfs = {key: trim_text(df) for key, df in dfs.items()}

In [0]:
# Replacing values
dfs["customer_crm"] = gender_fix(dfs["customer_crm"])
dfs["customer_erp"] = gender_fix(dfs["customer_erp"])
dfs["customer_crm"] = marital_status_fix(dfs["customer_crm"])
dfs["customer_location"] = location_fix(dfs["customer_location"])



In [0]:
# Renove duplicates and nulls
dfs = {key: df.dropDuplicates() for key, df in dfs.items()}
dfs = {key: df.dropna(how="any") for key, df in dfs.items()}

In [0]:
# Sales table date columns to actual dates (order_date, ship_date, due_date)

date_cols = ["order_date", "ship_date", "due_date"]

for column in date_cols:
    dfs["sales_crm"] = dfs["sales_crm"].withColumn(
        column, 
        try_to_date(col(column).cast("string"), "yyyyMMdd")
    )

## Custom transformations

### On Customers

In [0]:
# Removing hyphen from customer_key column in the customer location table.
dfs["customer_location"] = dfs["customer_location"].withColumn("customer_key", regexp_replace(col("customer_key"),"-",""))

In [0]:
# Seperating the product key and category key from the column "product_key" in the product_crm table
dfs["product_crm"] = dfs["product_crm"] \
    .withColumn("product_category_id", regexp_replace(substring(col("product_key"), 1, 5), "-", "_")) \
    .withColumn("product_key", substring(col("product_key"), 7, 7))

In [0]:
# Write to silver_staging
spark.sql("USE SCHEMA silver_staging")
for key, df in dfs.items():
    df.write.mode("overwrite").\
        option("overwriteSchema", "true").\
        saveAsTable(f"{key}")